# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Unit of analysis + time window

One row represents one anonymized content item (one page or article) in the starter dataset. The data is summarized around the recent performance window, especially the prior 30-day and last 30-day impressions windows, so the grain is `content_id` and the time window is the page’s recent 30-day performance history.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd


def load_refresh_data():
    base = Path.cwd()
    for candidate in [base, base.parent, base.parent.parent]:
        path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError("Could not find the starter dataset.")


df = load_refresh_data()
print("Rows, columns:", df.shape)
print("Unique content_id:", df["content_id"].nunique())
print("Unique client_id:", df["client_id"].nunique())
print("Sample window columns:")
print(df[["impressions_prev_30d", "impressions_last_30d", "impressions_90d"]].head())
print("No missing prev/last impressions:", df[["impressions_prev_30d", "impressions_last_30d"]].isna().sum().sum())

Rows, columns: (30000, 44)
Unique content_id: 30000
Unique client_id: 32
Sample window columns:
   impressions_prev_30d  impressions_last_30d  impressions_90d
0                   987                   578             3803
1                  5915                  2501            15320
2                  6089                  2382            12581
3                  4206                  3626            11751
4                  6452                  4211            19140
No missing prev/last impressions: 0


## 2. Fields: feature / label / context / excluded

Features:
- `search_volume`, `competition`, `competition_level`, `cpc`
- `content_type`, `main_intent`
- `word_count`, `char_count`, `content_age_days`, `days_since_last_update`
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- `impression_tier`, `position_tier`, `age_tier`, `word_count_tier`, `char_count_tier`

Label:
- `impressions_last_30d < impressions_prev_30d` as a decline proxy for future opportunity scoring.

Context:
- `content_id`, `client_id` to maintain the item identity and grouping.
- `trend_direction`, `trend_pct` as additional descriptive status signals.

Excluded:
- `clicks_last_30d`, `sessions_last_30d` because they come from the same future/label window and should not be used as training features.
- `impressions_90d` when strict label alignment is required, since using a full 90-day summary can mix the training window with the target window.
- Any identifiers or fields that leak editorial actions or private content details (not present in this starter dataset).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Feature list count:", 35)
print("Context fields:", ["content_id", "client_id", "trend_direction", "trend_pct"])
print("Label proxy definition:", "impressions_last_30d < impressions_prev_30d")
print("Excluded future-window fields:", ["clicks_last_30d", "sessions_last_30d"])
print("Impression 90d field excluded when strict label-window alignment is needed.")

Feature list count: 35
Context fields: ['content_id', 'client_id', 'trend_direction', 'trend_pct']
Label proxy definition: impressions_last_30d < impressions_prev_30d
Excluded future-window fields: ['clicks_last_30d', 'sessions_last_30d']
Impression 90d field excluded when strict label-window alignment is needed.


## 3. Verify it with queries (grain, counts, missing values, windows)
The grain is one `content_id` per row, and the dataset contains 30,000 unique content items. The time windows are represented by the `impressions_prev_30d` and `impressions_last_30d` columns, which are non-missing for all rows and support the decline proxy label.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("content_id unique:", df["content_id"].nunique())
print("rows:", len(df))
print("most common content_type:\n", df["content_type"].value_counts().head())
print("\nImpression window stats:")
print(df[["impressions_prev_30d", "impressions_last_30d"]].describe())
print("\nMissing trend_pct:", df["trend_pct"].isna().sum())
print("Trend direction values:\n", df["trend_direction"].value_counts())
print("\nOverlap check: prev/last windows seem defined by columns, not dates.")

content_id unique: 30000
rows: 30000
most common content_type:
 content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Impression window stats:
       impressions_prev_30d  impressions_last_30d
count          30000.000000          30000.000000
mean            1783.078500           1429.058733
std             6150.429511           5643.852081
min                0.000000              0.000000
25%               19.000000             10.000000
50%              210.000000            139.000000
75%             1143.000000            768.000000
max           218786.000000         238796.000000

Missing trend_pct: 3388
Trend direction values:
 trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Overlap check: prev/last windows seem defined by columns, not dates.


## 4. Data limits

This data can tell us which pages were declining recently, but it cannot prove that a refresh caused later gains. It does not include content text, URLs, editorial decisions, or the actual refresh actions taken on a page. The strongest use is decision support: ranking candidate pages to review, not claiming causal impact or guaranteed recovery.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("This data cannot tell us causal impact from refresh actions.")
print("It does not include content text, editorial decisions, or actual refresh events.")
print("It can tell us which pages were declining recently, but not whether a refresh would have fixed them.")
print("This is decision-support data, not an experiment or causal proof set.")

This data cannot tell us causal impact from refresh actions.
It does not include content text, editorial decisions, or actual refresh events.
It can tell us which pages were declining recently, but not whether a refresh would have fixed them.
This is decision-support data, not an experiment or causal proof set.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.